<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 2.2 — RAG Foundations: Embeddings + Retrieval + LLM (Optional)

In this notebook, you will build a simple Retrieval-Augmented Generation (RAG) workflow over de-identified clinical notes prepared in Lab 6.

### What you will do
1) Load cleaned notes (Lab 1 output)
2) Embed notes using MiniLM (local embeddings)
3) Use FAISS to retrieve the most relevant notes for a query
4) Provide retrieved notes to a local LLM (Ollama) to extract a **dementia phenotype (Yes/No)**

### Why this matters
Dementia is often under-coded in structured diagnosis tables. RAG lets us search narrative notes for phenotyping signals that structured data may miss.

## 1. Prepare Notes Data for RAG

We will load the cleaned notes dataset from Lab 6 (deduplicated and restricted to the cohort window).

**Important:**
We will NOT use physician gold labels inside the retrieval or prompt context (to avoid leakage).
Gold labels are only used later for evaluation.

In [ ]:
# -----------------------------------------------------------
# 1.1. Load Cleaned Notes + Evaluation Labels (Lab 6 Outputs)
# -----------------------------------------------------------
# Inputs:
# - lab6_clean_notes_baseline.parquet  (note-level dataset for RAG)
# - lab6_structured_dementia_eval.csv  (patient-level labels for evaluation only)
#
# Goal:
# - notes_df: used for embeddings + retrieval (NO gold labels attached)
# - eval_df: used later to evaluate RAG outputs
# -----------------------------------------------------------

from IPython.display import display, Markdown
from pathlib import Path
import pandas as pd

filepath = Path("data_files")

notes_df = pd.read_parquet(filepath / "lab1_clean_notes_baseline.parquet")
eval_df  = pd.read_csv(filepath / "lab1_structured_dementia_eval.csv")

print("Notes shape:", notes_df.shape)
print("Eval shape :", eval_df.shape)

display(notes_df.head(10))

## 2. Embed and Store Clinical Notes in a FAISS Vector Store (In-Memory)

In this step, we embed full clinical notes and store them in a **FAISS** (Facebook AI Similarity Search
) vector store, which enables efficient similarity search. We use a lightweight transformer model (`MiniLM`) to convert each note into a semantic vector.

### Steps:
- **2.1**: Embed clinical notes using the `all-MiniLM-L6-v2` model from Hugging Face.
- **2.2**: View an embedded document along with its metadata and vector representation.

### Why Use This Approach?

Storing entire notes is useful when:
- You want to preserve the full clinical context for each patient.
- Your downstream use case (e.g., summarization or structured extraction) requires complete narrative input.
- The notes are concise enough to fit within the input limits of an LLM.

This method simplifies retrieval workflows by allowing you to work with whole documents rather than fragmented chunks.

<img src="./images/rag_full.png" alt="RAG Full" width="900">




In [ ]:
# -----------------------------------------------------------
# 2.1. Embed Clinical Notes Using Local MiniLM Embeddings
# -----------------------------------------------------------
# This cell encodes each clinical note into a vector using a local
# transformer model and stores those embeddings in a FAISS index for
# fast similarity search.

# Model: `sentence-transformers/all-MiniLM-L6-v2`
# - Optimized for semantic similarity tasks
# - Lightweight and fast (384-dimensional vectors)
# - Runs fully offline
# -----------------------------------------------------------

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Prepare inputs for embedding: note text and relevant metadata
documents = notes_df["note_txt_deid"].tolist()
metadata = notes_df[["patient_num", "visit_id", "create_dts_shifted", "note_csn_id" ,"inpatient_note_type_dsc"]].to_dict(orient="records")

# Create a FAISS vector store from the documents
vectorstore = FAISS.from_texts(
    documents,
    embedding_model,
    metadatas=metadata
)

print(f"✅ Successfully embedded {len(documents)} clinical notes using MiniLM.")


In [ ]:
# -----------------------------------------------------------
# 2.2. View a Specific Embedded Document, Metadata, and Vector
# -----------------------------------------------------------
# Select an index (e.g., id = 5) to inspect the stored document.
# This cell shows the document text, associated metadata, and
# the corresponding FAISS embedding vector.
# -----------------------------------------------------------

id = 1  # You can change this index to view a different record

# Get (doc_id, Document) tuple from LangChain's docstore
doc_id, doc_example = list(vectorstore.docstore._dict.items())[id]

# Retrieve corresponding vector from FAISS
vector_example = vectorstore.index.reconstruct(id)

display(Markdown(f"### 🧾 Document ID: `{doc_id}`"))
display(Markdown(f"**Metadata:** `{doc_example.metadata}`"))

display(Markdown("**Document Text (First 500 characters):**"))
display(Markdown(f"```text\n{doc_example.page_content}...\n```"))

display(Markdown("**Embedded Vector (First 100):**"))
display(Markdown(f"```text\n{vector_example[:100]}\n```"))

## 3. Retrieving Clinical Notes with Similarity Score (RAG Retrieval)

In this section, we perform semantic search over embedded clinical notes using a FAISS vector store and a locally generated query vector. We use similarity scores to evaluate the relevance of each match to the query.

### Key Retrieval Steps:

1. **Embed a Query (Step 3.1)**
   - Converts a natural language question into a numerical vector using the same MiniLM model used to embed the notes.

2. **Similarity Search with Scores (Step 3.2)**
   - Retrieves the top-k clinical notes ranked by cosine similarity to the query.
   - Includes similarity scores for transparency and ranking.

3. **Score Threshold Filtering (Step 3.3)**
   - Filters out matches with low similarity scores.
   - Helps improve the precision and clinical relevance of the results.

### Why Use These Techniques?

Similarity search helps identify notes most relevant to a user-defined question or condition. Threshold filtering ensures:
- Only strong matches are considered for downstream tasks like summarization
- Noisy or unrelated content is excluded
- Each result can be justified based on a similarity score

<img src="./images/rag_retrieval.png" alt="RAG Retrieval" width="900">


In [ ]:
# -----------------------------------------------------------
# 3.1. Embed a Query and Inspect Its Vector Representation
# -----------------------------------------------------------
# This step encodes a natural language query into a numerical vector
# using the same MiniLM model used for the clinical notes.
# This vector will be used to search for semantically similar notes.
# -----------------------------------------------------------

# Define a sample clinical query
query = "Which patients have documented dementia or Alzheimer disease?"

# Generate the embedding for the query
query_vector = embedding_model.embed_query(query)

# Display the vector and its shape
display(Markdown("### Vectorized Query"))
display(Markdown(f"`Query:` *{query}*"))
display(Markdown("**Embedding Vector (truncated):**"))
display(Markdown(f"```text\n{query_vector[:100]} ... [{len(query_vector)} dimensions]\n```"))


In [ ]:
# -----------------------------------------------------------
# 3.2. Similarity Search (Top-K Results, No Filtering)
# -----------------------------------------------------------
# This cell performs a semantic similarity search using the embedded query,
# returning the top-k most similar clinical notes along with similarity scores.

# Score Interpretation (Euclidean Distance):
# - 0.00 – 0.30: Highly relevant
# - 0.30 – 0.50: Strong match
# - 0.50 – 0.70: Moderate match
# - 0.70 – 0.90: Low match
# - 0.90+: Minimal or irrelevant

# similarity_score = 1 / (1 + distance)  # Rescales Euclidean Distance into similarity-like score
# -----------------------------------------------------------

# Define number of top results
top_k = 10

# Run similarity search
results = vectorstore.similarity_search_with_score(query, k=top_k)

# Display header
display(Markdown(f"### 🔍 Top {top_k} Most Similar Clinical Notes"))

# Iterate and display each match
for i, (doc, score) in enumerate(results):
    display(Markdown(f"---\n**Result {i+1}**  \n- **Distance Score:** `{score:.4f}`  \n- **Patient Num:** `{doc.metadata.get('patient_num', 'N/A')}`  \n- **Visit ID:** `{doc.metadata.get('visit_id', 'N/A')}`\n\n**Note Preview:**\n```text\n{doc.page_content[:1200]}\n```"))


In [ ]:
# -----------------------------------------------------------
# 3.3. Filter Search Results by Similarity Score Threshold
# -----------------------------------------------------------
# This cell filters the top-K search results to keep only those
# with high similarity scores above a defined threshold.

# Similarity Score Threshold:
# - Only notes with scores ≥ threshold will be retained.
# - Higher scores = greater semantic similarity.
# -----------------------------------------------------------


from IPython.display import Markdown, display

threshold = 0.9  # Keep notes with score ≥ x

# Initialize an empty list to hold the filtered results
filtered_results = []

# Loop through each result (a tuple of Document and score)

for doc, distance in results:
    # Check if the similarity score meets the threshold
    if distance <= threshold:
        # If so, add it to the filtered list
        filtered_results.append((doc, distance))


# Summary
display(Markdown(f"### ✅ {len(filtered_results)} of {top_k} notes passed the distance threshold (<= {threshold})"))

# Show filtered results
for i, (doc, similarity) in enumerate(filtered_results):
    display(Markdown(
        f"---\n**Filtered Match {i+1}**  \n"
        f"- **Distance Score:** `{similarity:.4f}`  \n"
        f"- **Patient Num:** `{doc.metadata.get('patient_num', 'N/A')}`  \n"
        f"- **Visit ID:** `{doc.metadata.get('vist_id', 'N/A')}`\n\n"
        f"**Note Preview:**\n```text\n{doc.page_content[:1200]}\n```"
    ))


## 4. Generating Structured Responses with an LLM (RAG Generation)

In this section, we take the clinical notes retrieved via semantic search and pass them into a Large Language Model (LLM) to generate structured, clinically meaningful responses. This is the final step in the **Retrieval-Augmented Generation (RAG)** pipeline.

### Key Steps:

1. **Creating a Prompt Template for LLM Querying (Step 4.1)**
   - Defines a reusable prompt structure for analyzing and summarizing clinical notes.
   - Ensures each response includes patient metadata and clear, structured outputs.

2. **Invoking LLM with Retrieved Context (Step 4.2)**
   - Inserts the top-retrieved clinical notes into the prompt.
   - Sends the prompt to a local LLM (e.g., Qwen2 via Ollama) for structured generation.
   - Returns a summary that directly answers the user’s medical query.

### Why This Matters

This step demonstrates how LLMs can synthesize information from real patient notes to produce:
- Patient-specific summaries
- Answered clinical questions
- Traceable outputs with structured identifiers

This capability is essential for use cases like clinical decision support, patient-facing summaries, or intelligent search interfaces.

<img src="./images/rag_generation.png" alt="RAG Generation" width="1250">


In [ ]:
# -----------------------------------------------------------
# 4.1. Prompt Template: Dementia Phenotype Extraction (Yes/No)
# -----------------------------------------------------------
# We will ask the LLM to phenotype dementia using ONLY retrieved note text.
# No gold labels are included in the prompt.
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a clinical phenotyping assistant. "
     "Use ONLY the provided notes. Do NOT infer or invent. "
     "Do NOT mention privacy/redaction/date-shifting metadata."),
    ("human",
     "Retrieved notes:\n\n{retrieved_docs}\n\n"
     "Clinical question: {query}\n\n"
     "Return a structured answer using bullet points:\n"
     "1) Dementia Phenotype (Yes/No)\n"
     "2) Evidence: up to 3 short direct quotes/phrases from the notes (or 'None')\n"
     "3) Rationale: 1–2 sentences grounded in the evidence\n"
     "4) Confidence: low/medium/high\n"
     "\nRules for Dementia Phenotype:\n"
     "- Answer 'Yes' ONLY if dementia is explicitly documented (e.g., 'dementia', 'Alzheimer', "
     "'vascular dementia', 'Lewy body dementia') or clearly stated as a prior diagnosis.\n"
     "- If dementia is not explicitly mentioned, answer 'No' (do not infer from memory complaints alone)."
    )
])

print(">> Prompt created and ready to use.")

In [ ]:
# -----------------------------------------------------------
# 4.2. Run the Phenotype Prompt on ALL Filtered Retrieved Notes
# -----------------------------------------------------------
# We include identifiers (patient_num, visit_id, note_csn_id) for traceability.
# We pass all filtered results (after similarity threshold) into the prompt.
# -----------------------------------------------------------

from IPython.display import display, Markdown
from langchain_ollama import ChatOllama

# Safety: limit how much text per note we pass to the model
MAX_CHARS_PER_NOTE = 2500

if len(filtered_results) == 0:
    display(Markdown("### No notes passed the similarity threshold."))
else:
    # Build retrieved docs text with identifiers
    retrieved_blocks = []
    for rank, (doc, score) in enumerate(filtered_results, start=1):
        meta = doc.metadata or {}

        patient_num = meta.get("patient_num", "N/A")
        visit_id = meta.get("visit_id", "N/A")
        note_csn_id = meta.get("note_csn_id", "N/A")
        note_type = meta.get("inpatient_note_type_dsc", "N/A")
        create_dts = meta.get("create_dts_shifted", "N/A")

        note_text = (doc.page_content or "").strip()
        note_text = note_text[:MAX_CHARS_PER_NOTE]

        retrieved_blocks.append(
            f"---\n"
            f"[Retrieved Note #{rank} | distance={score:.4f}]\n"
            f"patient_num={patient_num} | visit_id={visit_id} | note_csn_id={note_csn_id}\n"
            f"note_type={note_type} | create_dts_shifted={create_dts}\n\n"
            f"{note_text}\n"
        )

    retrieved_context = "\n".join(retrieved_blocks)

    # Show what will be sent (optional but useful in workshop)
    display(Markdown(f"### Retrieved notes sent to the LLM: {len(filtered_results)}"))
    display(Markdown(retrieved_context[:3000] + ("\n\n*(preview truncated)*" if len(retrieved_context) > 3000 else "")))

    # Fill prompt
    final_prompt = prompt_template.format(
        retrieved_docs=retrieved_context,
        query=query
    )

    # Invoke local model (deterministic)
    model = ChatOllama(model="gemma3:12b", temperature=0.0)
    response = model.invoke(final_prompt)

    display(Markdown("### 📋 LLM Output: Dementia Phenotype from Retrieved Notes"))
    display(Markdown(response.content))